# Feature Engineering — IEEE-CIS Fraud Detection

**Objetivo de este notebook:** crear nuevas variables a partir de los hallazgos del análisis exploratorio, para que los modelos puedan detectar mejor el fraude.

En el notebook 01 comprobamos que ninguna variable explica el fraude por sí sola (la correlación máxima con el target es de 0,15), pero que hay señales claras en la ausencia de datos, en el importe, en algunas categorías y en las interacciones entre variables. En este notebook convertimos esas señales en variables:
- **Separación temporal** entre entrenamiento y validación, antes de crear cualquier variable que aprenda de los datos.
- **Variables por reglas fijas:** indicadores de ausencia de datos, familias de sistema operativo y navegador, marca del dispositivo, variables de tiempo y transformaciones del importe.
- **Variables agregadas:** resúmenes del comportamiento de cada tarjeta, calculados sin usar información del periodo de validación.
- **Comprobación:** relación de cada variable nueva con el fraude.

Guardamos los conjuntos de entrenamiento, validación y test con las nuevas variables. El tratamiento de nulos, el encoding y el escalado dependen de cada modelo, por lo que se aplicarán dentro de los pipelines del notebook de modelado.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid", palette="deep", rc={"figure.figsize": (10, 5)})

sorted(p.name for p in PROCESSED_DIR .glob("*.parquet"))

['test_clean.parquet',
 'test_fe.parquet',
 'train_clean.parquet',
 'train_fe.parquet',
 'valid_fe.parquet']

In [3]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")

print(train.shape)
print(test.shape)

(590537, 434)
(506691, 433)


## 1. Split

In [4]:
corte = train["TransactionDT"].quantile(0.8)
train["split"] = np.where(train["TransactionDT"] <= corte, "train", "valid")
test["split"] = "test"

df = pd.concat([train, test], ignore_index=True)
del train, test

df["split"].value_counts()

split
test     506691
train    472429
valid    118108
Name: count, dtype: int64

**Split temporal**

Separamos los datos por fecha y no al azar: las transacciones más antiguas (el 80 % inferior de `TransactionDT`) forman el conjunto de **entrenamiento** y el 20 % más reciente el de **validación**. Así validamos el modelo igual que se usaría en producción, prediciendo el futuro a partir del pasado. Un split aleatorio mezclaría transacciones de todo el periodo y daría una estimación demasiado optimista.

Las transacciones de test se marcan como `test` y se unen todas en un único DataFrame `df`, de modo que cada transformación se escribe una sola vez y se aplica por igual a los tres conjuntos. Cuando una variable necesita aprender algo de los datos (por ejemplo, qué categorías son frecuentes), lo calculamos solo con las filas de `train`, para no filtrar información de validación ni de test.

## 2. Nulos

In [5]:
df['has_identity'] = df['id_12'].notna()
df['has_addr']= df['addr1'].notna()
df['has_r_email'] = df['R_emaildomain'].notna()



En el análisis exploratorio vimos que los nulos de muchas columnas no son aleatorios, pero que cada modelo los gestiona de forma distinta (los árboles de boosting los tratan de forma nativa, la regresión logística necesita imputarlos), por lo que su tratamiento general se hará dentro de los pipelines del notebook de modelado.

Aquí solo creamos tres indicadores de presencia, porque la ausencia de estos datos sí mostraba una relación clara con el fraude:
- `has_identity`: la transacción tiene datos de identidad (`id_12` no es nulo).
- `has_addr`: tiene dirección (`addr1` no es nulo).
- `has_r_email`: tiene email del destinatario (`R_emaildomain` no es nulo).

In [6]:
df["os_family"] = df["id_30"].str.split().str[0]

In [7]:
print(df["os_family"].unique())
df[df["split"] == "train"].groupby("os_family", dropna=False)["isFraud"].mean()

[nan 'android' 'ios' 'mac' 'windows' 'linux' 'func' 'other']


os_family
android   0.086
func      0.000
ios       0.061
linux     0.082
mac       0.018
other     0.333
windows   0.035
NaN       0.034
Name: isFraud, dtype: float64

## Familia del sistema operativo

`id_30` contiene la versión exacta del sistema operativo (`windows 10`, `ios 11.4.1`, `mac os x 10_14_6`…). Hay decenas de versiones y muchas tienen muy pocas transacciones, así que el modelo no puede aprender un patrón fiable de cada una. Agrupamos las versiones en familias quedándonos con la primera palabra, de modo que cada grupo reúne suficientes filas.

Calculamos la tasa de fraude por familia **solo con las filas de entrenamiento**, para no usar información de validación ni de test:

- **Android (8,6%), Linux (8,2%) e iOS (6,1%)** superan con claridad la tasa general del 3,5%.
- **Windows (3,5%)** está en la media y **Mac (1,8%)** queda por debajo.
- **`other` (33,3%) y `func` (0%)** muestran valores extremos, pero se calculan sobre muy pocas filas (9 y 10), por lo que no son fiables. `func` es un valor sin documentar en el dataset original.

**Conclusión:** la familia del sistema operativo separa grupos con riesgo claramente distinto: Android, Linux e iOS concentran más fraude que Windows y Mac. Es la primera señal de un patrón que se repite en las variables siguientes, el de los dispositivos móviles con más fraude que los de escritorio.

In [8]:
df["browser_family"] = df["id_31"].str.split().str[0]
df["browser_family"] = df["browser_family"].replace({"mobile": "safari", "mozilla/firefox": "firefox"})

counts = df[df["split"] == "train"]["browser_family"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["browser_family"].isin(keep) & df["browser_family"].notna(), "browser_family"] = "other"

df[df["split"] == "train"].groupby("browser_family", dropna=False)["isFraud"].agg(["mean", "count"])
  

,mean,count
browser_family,,
chrome,0.093,63376
edge,0.034,4614
firefox,0.071,6059
ie,0.018,8454
other,0.228,1300
safari,0.062,31994
samsung,0.083,1666
NaN,0.021,354966


## Familia del navegador

`id_31` contiene el navegador y su versión (`chrome 63.0`, `mobile safari 11.0`…). Igual que con el sistema operativo, nos quedamos con la primera palabra para agrupar las versiones en familias, y corregimos los dos casos en los que esa regla falla: `mobile safari` pasa a `safari` y `mozilla/firefox` a `firefox`.

Muchas familias tienen muy pocas transacciones y tasas de fraude extremas (0% o 100%) que son ruido, no señal. Para evitar que el modelo aprenda esas casualidades (overfitting), **conservamos solo las familias con al menos 500 transacciones en entrenamiento**, que con una tasa base del 3,5% equivalen a unos 17 fraudes esperados. El resto se agrupa en `other`. Al definir la lista de familias que conservamos, y no la de las raras, los navegadores que aparezcan por primera vez en test también se asignan a `other`.

Resultados en entrenamiento:

- **`other` (22,8%)** tiene la tasa más alta. Incluye navegadores poco comunes de móviles Android, que por separado ya mostraban tasas muy elevadas.
- **Chrome (9,3%), Samsung (8,3%), Firefox (7,1%) y Safari (6,2%)** están por encima de la media del 3,5%. Destaca Chrome, por combinar una tasa casi tres veces superior a la media con el mayor volumen de transacciones.
- **Edge (3,4%) e Internet Explorer (1,8%)** están en la media o por debajo.
- Las transacciones **sin navegador (2,1%)** coinciden con las que no tienen datos de identidad.

**Conclusión:** la familia del navegador separa grupos con riesgo claramente distinto y refuerza lo visto con el sistema operativo: los dispositivos móviles concentran más fraude que los de escritorio.

In [9]:
df["device_brand"] = df["DeviceInfo"].str.split().str[0]
df.loc[df["device_brand"].str.startswith("sm-", na=False), "device_brand"] = "samsung"
df.loc[df["device_brand"].str.startswith(("rv:", "trident"), na=False), "device_brand"] = "desktop_other"

counts = df[df["split"] == "train"]["device_brand"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["device_brand"].isin(keep) & df["device_brand"].notna(), "device_brand"] = "other"

df[df["split"] == "train"].groupby("device_brand", dropna=False)["isFraud"].agg(["mean", "count"])

,mean,count
device_brand,,
desktop_other,0.034,10263
huawei,0.101,605
ios,0.061,17083
macos,0.019,10843
moto,0.163,2103
other,0.139,9459
samsung,0.118,9458
windows,0.062,39722
NaN,0.026,372893


## Marca del dispositivo

`DeviceInfo` identifica el dispositivo con el que se hizo la compra. Aplicamos la misma técnica que con el navegador: primera palabra y agrupación de las marcas con menos de 500 transacciones en `other`.

Antes de agrupar, corregimos dos casos en los que la regla de la primera palabra no funciona:

- **Modelos de Samsung:** cada modelo aparece con su propio código (`sm-j700m`, `sm-g610m`…). Por separado, casi ninguno llega a 500 transacciones, por lo que la mayoría acababan en `other` en lugar de contarse como Samsung. Asignamos a `samsung` todos los valores que empiezan por `sm-`. Con esta corrección, Samsung pasa de 1.900 a 9.458 transacciones.
- **Identificadores de navegador:** `rv:11.0`, `rv:57.0` y `trident/7.0` no son marcas, sino fragmentos técnicos del navegador de un ordenador (Internet Explorer y Firefox). Esa información ya está en `browser_family`, así que los agrupamos en `desktop_other`.

Resultados en entrenamiento:

- **Motorola (16,3%), `other` (13,9%), Samsung (11,8%) y Huawei (10,1%)**, todas marcas de móviles Android o modelos poco comunes, triplican o más la media del 3,5%.
- **iOS (6,1%) y Windows (6,2%)** están por encima de la media.
- **`desktop_other` (3,4%)** está en la media y **macOS (1,9%)** queda por debajo.
- Las transacciones **sin dispositivo (2,6%)** coinciden con las que no tienen datos de identidad.

**Conclusión:** la marca del dispositivo confirma el patrón de las variables anteriores: los móviles, especialmente los Android, concentran más fraude que los ordenadores.

In [10]:
df["p_email_provider"] = df["P_emaildomain"].str.split(".").str[0]

counts = df[df["split"] == "train"]["p_email_provider"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["p_email_provider"].isin(keep) & df["p_email_provider"].notna(), "p_email_provider"] = "other"

df[df["split"] == "train"].groupby("p_email_provider", dropna=False)["isFraud"].agg(["mean", "count"])

,mean,count
p_email_provider,,
anonymous,0.023,29961
aol,0.022,22618
att,0.006,3133
bellsouth,0.028,1580
charter,0.023,688
comcast,0.030,6496
cox,0.021,1121
gmail,0.044,183101
hotmail,0.052,37736


## Proveedor del email del comprador

`P_emaildomain` contiene el dominio del email del comprador. Agrupamos los dominios por proveedor quedándonos con la parte anterior al primer punto (`yahoo.com` y `yahoo.com.mx` pasan a `yahoo`), ya que el país del dominio dispersa las transacciones en grupos demasiado pequeños. Después aplicamos la misma agrupación en `other` de los proveedores con menos de 500 transacciones.

Resultados en entrenamiento:

- **Outlook (9,5%)** casi triplica la media del 3,5%, y **Hotmail (5,2%)** también la supera. Son cuentas gratuitas que se crean y se abandonan con facilidad.
- **Los proveedores de internet de EE. UU.** (`att`, `sbcglobal`, `verizon`, `optonline`, entre el 0,5% y el 1%) tienen entre 4 y 7 veces menos fraude que la media. Estos emails están ligados a un contrato real con el proveedor.
- **Gmail (4,4%)**, que concentra el mayor volumen, y **`anonymous` (2,3%)** están cerca de la media.

**Conclusión:** la señal es más débil que la del dispositivo o el navegador, pero el proveedor permite identificar tanto grupos de riesgo como grupos especialmente seguros, así que lo incluimos como variable.

In [11]:
df["r_email_provider"] = df["R_emaildomain"].str.split(".").str[0]

counts = df[df["split"] == "train"]["r_email_provider"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["r_email_provider"].isin(keep) & df["r_email_provider"].notna(), "r_email_provider"] = "other"

df[df["split"] == "train"].groupby("r_email_provider", dropna=False)["isFraud"].agg(["mean", "count"])

,mean,count
r_email_provider,,
anonymous,0.029,17271
aol,0.037,3254
comcast,0.013,1578
gmail,0.115,47370
hotmail,0.073,23548
icloud,0.119,1138
live,0.037,1329
msn,0.001,718
other,0.027,4064


## Proveedor del email del destinatario

`R_emaildomain` contiene el dominio del email del destinatario de la compra. Le aplicamos el mismo tratamiento que al email del comprador: nos quedamos con la parte anterior al primer punto (`yahoo.com` y `yahoo.com.mx` pasan a `yahoo`) y agrupamos en `other` los proveedores con menos de 500 transacciones en entrenamiento. Después eliminaremos la columna original, ya que su información queda recogida en `r_email_provider`.

Esta variable complementa a `has_r_email`: el indicador dice si existe email del destinatario, y el proveedor, de qué tipo es.

Resultados en entrenamiento:

- **Outlook (15,3%), iCloud (11,9%) y Gmail (11,5%)** multiplican por entre tres y cuatro la media del 3,5%. Destaca Gmail, por combinar una tasa alta con más de 47.000 transacciones.
- **Hotmail (7,3%) y Yahoo (5,2%)** también superan la media.
- **MSN (0,1%), Verizon (0%) y Comcast (1,3%)** apenas tienen fraude. Como en el email del comprador, los proveedores de internet de EE. UU. son los más seguros.
- Las transacciones **sin email del destinatario (2,1%)** están por debajo de la media.

**Conclusión:** el proveedor del email del destinatario separa el fraude con más claridad que el del comprador: Gmail pasa del 4,4% como comprador al 11,5% como destinatario. Tiene sentido, ya que el destinatario es quien recibe el producto, y en un fraude suele ser una cuenta controlada por el defraudador.

In [12]:
counts = df[df["split"] == "train"]["id_33"].value_counts()
keep = counts[counts >= 500].index
df.loc[~df["id_33"].isin(keep) & df["id_33"].notna(), "id_33"] = "other"

(df[df["split"] == "train"]
 .groupby("id_33", dropna=False)["isFraud"]
 .agg(["mean", "count"])
 .sort_values("count", ascending=False))

,mean,count
id_33,,
NaN,0.034,410616
1920x1080,0.029,14460
1366x768,0.065,6669
1334x750,0.059,5602
other,0.067,4846
2208x1242,0.095,4166
1440x900,0.024,3792
2048x1536,0.017,3110
1600x900,0.021,3098


## Resolución de pantalla

`id_33` contiene la resolución de pantalla del dispositivo (`1920x1080`, `1366x768`…), con más de 200 valores distintos. Agrupamos en `other` las resoluciones con menos de 500 transacciones en entrenamiento, con el mismo criterio que en las variables anteriores. En este caso no creamos una columna nueva: la resolución ya es la categoría, así que solo agrupamos las raras.

Resultados en entrenamiento:

- **1280x720 (14,6%) y 2436x1125 (12,5%)** cuadruplican la media del 3,5%, y **2208x1242 (9,5%)** casi la triplica. Son resoluciones de móvil: la segunda corresponde al iPhone X y la tercera, a los iPhone Plus.
- **`other` (6,7%) y 1366x768 (6,5%)**, la resolución típica de los portátiles más económicos, también superan la media.
- **Las resoluciones altas de ordenador** quedan por debajo: 2560x1440 (0,5%), 1680x1050 (1,6%), 2048x1536 (1,7%) y las de MacBook, 2880x1800 (1,9%) y 1440x900 (2,4%).
- **1920x1080 (2,9%)**, la más frecuente, está cerca de la media.
- Las transacciones **sin resolución (3,4%)** son la gran mayoría (más de 410.000) y están en la media.

**Conclusión:** la resolución refuerza el patrón de las variables de dispositivo: las pantallas de móvil concentran más fraude, y las de ordenador de gama alta, menos.

In [13]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["weekday"] = (df["TransactionDT"] // 86400) % 7

df[df["split"] == "train"].groupby("hour")["isFraud"].agg(["mean", "count"])

,mean,count
hour,,
0,0.032,30557
1,0.032,26772
2,0.037,21922
3,0.036,17102
4,0.050,12603
5,0.067,8392
6,0.074,5065
7,0.101,3156
8,0.097,2181


## Variables de tiempo

`TransactionDT` está en segundos desde un momento de referencia que Vesta no revela, así que no la convertimos a fecha: cualquier fecha de inicio sería una suposición. Calculamos la posición dentro del ciclo con división entera y resto:

- `hour = (TransactionDT // 3600) % 24`
- `weekday = (TransactionDT // 86400) % 7`

No creamos el año ni el mes: los datos cubren unos 6 meses y el test es posterior, por lo que sus valores no aparecerían nunca en entrenamiento. La hora y el día de la semana sí se repiten en ambos conjuntos.

Como no conocemos la hora de inicio, la hora 0 no es necesariamente medianoche. Aun así, la variable es interpretable gracias al volumen de transacciones:

- Las horas **7-9** tienen el **menor volumen** (2.000-3.000 transacciones, frente a más de 33.000 en las horas punta), así que corresponden a la madrugada real.
- En esas mismas horas el fraude alcanza su **máximo (10-10,6%)**, el triple de la media del 3,5%.
- Las horas **13-15** son las de **menor fraude** (2,1-2,5%).

**Conclusión:** el fraude se concentra en las horas de menor actividad, cuando hay menos compras legítimas que lo diluyan. La hora del día es una variable útil para el modelo.

In [14]:
df[df["split"] == "train"].groupby("weekday")["isFraud"].agg(["mean", "count"])

,mean,count
weekday,,
0,0.038,68635
1,0.035,80059
2,0.037,64926
3,0.037,55840
4,0.032,68191
5,0.033,66991
6,0.034,67787


El **día de la semana** no muestra diferencias (entre el 3,2% y el 3,8%, todos cerca de la media). La mantenemos por ahora, ya que podría aportar en combinación con otras variables, como la hora. Decidiremos si se conserva en la fase de modelado, a partir de la importancia que le asigne el modelo.

In [15]:
df["amt_cents"] = ((df["TransactionAmt"] % 1) * 100).round().astype("int8")

(df[df["split"] == "train"]
 .groupby("amt_cents")["isFraud"]
 .agg(["mean", "count"])
 .sort_values("count", ascending=False)
 .head(15))

,mean,count
amt_cents,,
0,0.036,244974
95,0.007,132133
50,0.020,17878
97,0.020,6570
99,0.023,4887
47,0.025,1814
36,0.092,1169
71,0.130,1014
43,0.068,965


## Céntimos del importe

Extraemos la parte decimal del importe en céntimos (`amt_cents`). Para el modelo, `49.99` y `50.00` son casi el mismo valor, pero los céntimos pueden revelar el tipo de compra: un precio de tienda, una cifra redonda o un importe convertido desde otra moneda. Un árbol necesitaría cientos de cortes en `TransactionAmt` para captar esta diferencia; con `amt_cents`, le basta uno.

Resultados en entrenamiento (15 valores más frecuentes):

- **`,95` (0,7%)**, con más de 130.000 transacciones, tiene cinco veces menos fraude que la media del 3,5%. Otros precios de tienda (`,50`, `,97`, `,99`) también quedan por debajo (2,0-2,3%).
- **`,00` (3,6%)**: las cifras redondas están en la media.
- **Céntimos irregulares** (`,11`, `,36`, `,53`, `,71`…) multiplican la media por tres o cuatro (9-14%). Probablemente corresponden a importes convertidos desde otra moneda, es decir, a compras internacionales.

**Conclusión:** los céntimos separan compras de riesgo muy distinto, una información que el importe por sí solo no hace accesible al modelo.

In [16]:
df[["card1", "card2", "card3", "card4", "card5", "card6"]].nunique()

card1    17091
card2      501
card3      133
card4        4
card5      138
card6        4
dtype: int64

In [17]:
train_rows = df[df["split"] == "train"]

card_counts = train_rows["card1"].value_counts()
card_means = train_rows.groupby("card1")["TransactionAmt"].mean()

df["card_count"] = df["card1"].map(card_counts).fillna(0)
df["card_amt_mean"] = df["card1"].map(card_means)
df["amt_ratio"] = df["TransactionAmt"] / df["card_amt_mean"]

In [18]:
ratio_bins = pd.cut(df["amt_ratio"], bins=[0, 0.5, 1, 2, 5, 1000])
df[df["split"] == "train"].groupby(ratio_bins[df["split"] == "train"])["isFraud"].agg(["mean", "count"])

,mean,count
amt_ratio,,
"(0.0, 0.5]",0.027,179436
"(0.5, 1.0]",0.034,154419
"(1.0, 2.0]",0.046,92950
"(2.0, 5.0]",0.054,38034
"(5.0, 1000.0]",0.035,7590


## Agregados por tarjeta

Un banco no solo evalúa una compra por sí misma, sino comparándola con el comportamiento habitual de la tarjeta. Para darle al modelo ese contexto, creamos tres variables por tarjeta.

Usamos `card1` como identificador de la tarjeta porque es la columna `card` con más valores distintos (unos 17.000). Las demás tienen entre 4 y 500, por lo que describen características compartidas por muchas tarjetas, como el tipo, el banco o el país.

- **`card_count`**: número de transacciones de la tarjeta.
- **`card_amt_mean`**: importe medio de la tarjeta.
- **`amt_ratio`**: importe de la transacción dividido entre el importe medio de su tarjeta. Un valor de 1 indica un gasto habitual; un valor de 5, un gasto cinco veces superior a lo normal.

**Evitar el leakage:** los agregados se calculan **solo con las filas de entrenamiento** y después se asignan a todas las filas con `.map()`. Validación y test son posteriores en el tiempo, así que usarlos para calcular las medias supondría usar compras que todavía no habían ocurrido. Las tarjetas que no aparecen en entrenamiento quedan con `card_count = 0` y con `card_amt_mean` y `amt_ratio` nulos.

In [19]:
safe_cents = [0, 95, 50, 97, 99, 47]

df["risk_score"] = ((df["has_identity"] == 1).astype("int8")
                  + (df["has_addr"] == 0).astype("int8")
                  + (df["has_r_email"] == 1).astype("int8")
                  + df["device_brand"].isin(["moto", "other", "samsung", "huawei"]).astype("int8")
                  + df["browser_family"].isin(["other", "chrome", "samsung"]).astype("int8")
                  + (~df["amt_cents"].isin(safe_cents)).astype("int8"))

df[df["split"] == "train"].groupby("risk_score")["isFraud"].agg(["mean", "count"])

,mean,count
risk_score,,
0,0.021,335179
1,0.010,17335
2,0.045,39047
3,0.051,28219
4,0.086,18371
5,0.123,19253
6,0.148,15025


## Riesgo compuesto

Ninguna variable explica el fraude por sí sola, pero varias señales moderadas pueden reforzarse entre sí. Creamos `risk_score`, que cuenta cuántas condiciones de riesgo cumple cada transacción, escritas de forma que `True` indique más riesgo:

1. Tiene datos de identidad (`has_identity == 1`).
2. No tiene dirección (`has_addr == 0`).
3. Tiene email del destinatario (`has_r_email == 1`).
4. El dispositivo es Motorola, Samsung, Huawei u `other`.
5. El navegador es Chrome, Samsung u `other`.
6. Los céntimos del importe no son de los habituales (`,00`, `,95`, `,50`, `,97`, `,99`, `,47`).

Las condiciones 1, 4 y 5 están relacionadas, ya que solo las transacciones con identidad tienen dispositivo y navegador.

| Condiciones | Tasa de fraude | Transacciones |
|---|---|---|
| 0 | 2,1% | 335.179 |
| 1 | 1,0% | 17.335 |
| 2 | 4,5% | 39.047 |
| 3 | 5,1% | 28.219 |
| 4 | 8,6% | 18.371 |
| 5 | 12,3% | 19.253 |
| 6 | 14,8% | 15.025 |

**Conclusión:** a partir de dos condiciones, el fraude aumenta en cada escalón, y las transacciones que cumplen las seis tienen siete veces más fraude que las que no cumplen ninguna. El nivel 1 es una excepción (1,0%): alguna condición, cuando aparece sola, se asocia a transacciones especialmente seguras.

## Sustitución de las columnas originales

In [20]:
df = df.drop(columns=["id_30", "id_31", "DeviceInfo", "P_emaildomain", "R_emaildomain"])

Eliminamos `id_30`, `id_31`, `DeviceInfo` y `P_emaildomain`, ya que su información está recogida en sus versiones agrupadas (`os_family`, `browser_family`, `device_brand` y `p_email_provider`). Conservarlas añadiría información duplicada y el riesgo de que el modelo aprenda de categorías con muy pocas transacciones.

## Codificación de categóricas

Los modelos necesitan valores numéricos, así que codificamos las 31 columnas de texto. El método depende del tipo de cada variable:

- **Binarias** (dos valores, como `found` / `notfound` o `t` / `f`): las convertimos en 0 y 1 con `.map`, indicando si la condición se cumple o no. Incluye `M1`–`M3`, `M5`–`M9`, las variables de identidad binarias y `DeviceType`.
- **Ordinales** (con un orden real): `id_34` (`match_status:-1` a `2`) se codifica respetando su orden.
- **Nominales** (sin orden entre categorías): usamos one-hot encoding, una columna 0/1 por categoría. Incluye `ProductCD`, `card4`, `card6`, `id_15`, `id_23`, `M4`, `id_33` y las familias creadas en este notebook.

Ninguna variable nominal supera las 21 categorías, porque las de mayor cardinalidad ya las agrupamos en las secciones anteriores. Por eso no necesitamos target encoding, que solo compensa con cientos de categorías y exige suavizado para evitar el leakage.

Los nulos se mantienen: en las binarias y en la ordinal quedan como `NaN`, y en el one-hot forman una categoría propia, ya que su ausencia aporta información.

### Variables binarias

In [21]:
binary_map = {"t": 1, "f": 0, "found": 1, "notfound": 0, "new": 0}
binary_cols = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9",
               "id_12", "id_16", "id_27", "id_28", "id_29",
               "id_35", "id_36", "id_37", "id_38"]

for col in binary_cols:
    df[col] = df[col].map(binary_map)

df["DeviceType"] = df["DeviceType"].map({"mobile": 1, "desktop": 0})


Las variables con dos valores se convierten en 0 y 1 con `.map`. Usamos el mismo criterio en todas para que el 1 signifique siempre lo mismo: verdadero (`t`) o encontrado (`found`). En `id_28`, cuyos valores son `new` y `found`, `found` es 1 y `new` es 0. En `DeviceType`, el 1 indica móvil y el 0, ordenador.

Los nulos se mantienen como `NaN`: los modelos de boosting los gestionan de forma nativa, y en la regresión logística se imputarán dentro de su pipeline.

### Variable ordinal

In [22]:
df["id_34"] = df["id_34"].map({"match_status:-1": -1, "match_status:0": 0,
                               "match_status:1": 1, "match_status:2": 2})

df["id_34"].value_counts(dropna=False)

id_34
NaN       947248
2.000     132185
1.000      17377
0.000        415
-1.000         3
Name: count, dtype: int64

`id_34` toma los valores `match_status:-1` a `match_status:2`, que indican un nivel. Conservamos el número de cada nivel para mantener su orden y su significado original.

### Variables nominales

In [23]:
onehot_cols = ["ProductCD", "card4", "card6", "M4", "id_15", "id_23", "id_33",
               "os_family", "browser_family", "device_brand",
               "p_email_provider", "r_email_provider"]

df = pd.get_dummies(df, columns=onehot_cols, dummy_na=True, dtype="int8")

df.select_dtypes(include=["object", "string"]).columns

Index(['split'], dtype='str')

In [24]:
remaining = df.select_dtypes(include=["object", "string"]).columns.drop("split")
print(len(remaining), list(remaining))

0 []


Las variables sin orden se codifican con one-hot encoding: cada categoría pasa a ser una columna 0/1. Incluimos una columna para los nulos (`dummy_na=True`), ya que su ausencia aporta información. Como las variables de mayor cardinalidad ya están agrupadas, el one-hot añade alrededor de un centenar de columnas.

## Separación y guardado

In [25]:
train = df[df["split"] == "train"].drop(columns="split")
valid = df[df["split"] == "valid"].drop(columns="split")
test = df[df["split"] == "test"].drop(columns=["split", "isFraud"])

train["isFraud"] = train["isFraud"].astype("int8")
valid["isFraud"] = valid["isFraud"].astype("int8")

train.to_parquet(PROCESSED_DIR / "train_fe.parquet", index=False)
valid.to_parquet(PROCESSED_DIR / "valid_fe.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test_fe.parquet", index=False)

print(train.shape, valid.shape, test.shape)

(472429, 541) (118108, 541) (506691, 540)


Una vez creadas todas las variables, separamos de nuevo los tres conjuntos usando la columna `split`, que ya no es necesaria y eliminamos. En test eliminamos también `isFraud`, que se añadió vacía al unir las tablas. Guardamos los resultados en Parquet con el sufijo `_fe` para el notebook de modelado.

## Conclusiones

Hemos creado 15 variables nuevas a partir de los hallazgos del análisis exploratorio:

| Técnica | Variables |
|---|---|
| Indicadores de presencia | `has_identity`, `has_addr`, `has_r_email` |
| Extracción y agrupación de categorías | `os_family`, `browser_family`, `device_brand`, `p_email_provider`, `r_email_provider` |
| Tiempo | `hour`, `weekday` |
| Extracción de un número | `amt_cents` |
| Agregados por tarjeta | `card_count`, `card_amt_mean`, `amt_ratio` |
| Riesgo compuesto | `risk_score` |

Además, agrupamos las resoluciones de pantalla poco frecuentes de `id_33`.

**Principales hallazgos:**

- **Los móviles concentran más fraude que los ordenadores:** dispositivos Android como Motorola (16,3%), Samsung (11,8%) o Huawei (10,1%), frente a macOS (1,9%).
- **El email del destinatario separa el fraude mejor que el del comprador:** Gmail pasa del 4,4% como comprador al 11,5% como destinatario, ya que el destinatario suele ser una cuenta controlada por el defraudador.
- **El fraude se concentra en las horas de menor actividad**, que corresponden a la madrugada real, con tasas de hasta el 10,6%.
- **Los céntimos irregulares**, probablemente compras con conversión de moneda, triplican o cuadruplican la media, mientras que los precios de tienda (`,95`) tienen cinco veces menos fraude.
- **Gastar más de lo habitual en una tarjeta aumenta el riesgo**: del 2,7% en los gastos más bajos al 5,4% cuando la compra es de 2 a 5 veces la media de la tarjeta.
- **Las señales se refuerzan entre sí:** las transacciones que cumplen las seis condiciones de riesgo tienen una tasa de fraude del 14,8%, **siete veces más** que las que no cumplen ninguna (2,1%).

**Preparación para el modelado:**

- Eliminamos las columnas originales de alta cardinalidad (`id_30`, `id_31`, `DeviceInfo`, `P_emaildomain` y `R_emaildomain`), sustituidas por sus versiones agrupadas.
- Codificamos todas las categóricas: las binarias como 0/1, `id_34` de forma ordinal y el resto con one-hot encoding, manteniendo los nulos como información.

**Decisiones para evitar el leakage:**

- Todo lo que aprende de los datos (el umbral de las categorías raras y los agregados por tarjeta) se calcula **solo con las filas de entrenamiento**.
- No creamos el mes ni el año, porque sus valores en test no aparecen en entrenamiento.

**Próximo paso:** en el notebook `03_modeling` imputaremos los nulos y escalaremos dentro del pipeline de la regresión logística, compararemos varios modelos con PR-AUC y decidiremos qué variables conservar según su importancia.